# Exploring the collate function

Before plugging `make_collate_fn` into a `DataLoader` and a Deepchecks `VisionData` object, let's see exactly what it produces for a single batch.

It takes a batch of raw Hugging Face dataset examples and returns a `BatchOutputFormat` with three things:
- `images`: a list of numpy arrays (H, W, C)
- `labels`: the ground-truth label for each image
- `predictions`: the model's predicted class probabilities for each image

In [ ]:
from datasets import load_dataset
from transformers import pipeline

from validate import make_collate_fn

dataset = load_dataset("beans", split="validation")
label_names = dataset.features["labels"].names
label_names

We use the base pretrained model here just to inspect the collate function. In `validate.py`, the pipeline comes from the `@dev` model loaded from the MLflow registry.

In [ ]:
pipe = pipeline("image-classification", model="nateraw/vit-base-beans")
collate_fn = make_collate_fn(pipe, label_names)

In [ ]:
dataset[0]

In [ ]:
batch = [dataset[i] for i in range(4)]
output = collate_fn(batch)

print("Number of images:", len(output["images"]))
print("Image shape:", output["images"][0].shape)
print("Labels:", output["labels"])
print("Predictions (probabilities per class):")
for p in output["predictions"]:
    print("  ", [round(x, 3) for x in p])

Let's look at one image alongside its true label and the model's predicted label.

In [ ]:
import matplotlib.pyplot as plt

idx = 0
true_label = label_names[output["labels"][idx]]
pred_label = label_names[output["predictions"][idx].index(max(output["predictions"][idx]))]

plt.imshow(output["images"][idx])
plt.title(f"True: {true_label} | Predicted: {pred_label}")
plt.axis("off")
plt.show()